[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/02_Boyles_Law.ipynb)

# DiveLab

## Notebook 02 — Boyle's Law

**Guiding question:** What happens to a gas volume when ambient pressure changes?

*Pressure changes linearly with depth. Gas volume does not.*

## Learning objectives

By the end of this lab, you will be able to:

- explain Boyle's law for a gas at constant temperature;
- connect ambient pressure to gas-volume changes during descent and ascent;
- calculate relative gas volume at different depths;
- visualize the nonlinear relationship between depth and gas volume;
- explain why the largest relative volume changes occur near the surface.

## From Notebook 01 to Notebook 02

In Notebook 01 we found that ambient pressure increases approximately linearly with depth.

A useful diving approximation is:

- 0 m → 1 bar
- 10 m → 2 bar
- 20 m → 3 bar
- 30 m → 4 bar
- 40 m → 5 bar

But we also noticed something important:

> The **relative** pressure change is largest near the surface.

Now we ask:

> What does this imply for a gas-filled space?

## Physical intuition

Imagine a flexible sealed air space.

When the surrounding pressure increases, the gas is compressed.

When the surrounding pressure decreases, the gas expands.

Examples in diving include gas spaces in:

- a BCD;
- a dry suit;
- a mask;
- the lungs during normal breathing.

For this notebook, we use an idealized gas model and assume temperature remains constant.

## Boyle's law

For a fixed amount of gas at constant temperature:

$$
PV = \text{constant}
$$

Therefore, between two equilibrium states:

$$
P_1V_1 = P_2V_2
$$

and

$$
V_2 = V_1 \frac{P_1}{P_2}
$$

## Important: use absolute pressure

Boyle's law must use **absolute pressure**, not gauge pressure.

At the surface, absolute pressure is approximately 1 bar.

At 10 m in seawater, absolute pressure is approximately 2 bar.

So a gas volume that is 1.0 unit at the surface becomes approximately:

$$
V = \frac{1}{2}
$$

of its original volume at 10 m.

## A normalized model

Let the gas volume at the surface be:

$$
V_0 = 1
$$

If surface pressure is $P_0$, then:

$$
P_0V_0 = P(z)V(z)
$$

so:

$$
V(z)=V_0\frac{P_0}{P(z)}
$$

If $V_0=1$, then $V(z)$ directly represents the **fraction of the surface volume**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Reuse the pressure model

We use the same physical constants as in Notebook 01.

In [ ]:
rho = 1025.0       # seawater density [kg/m^3]
g = 9.80665        # gravitational acceleration [m/s^2]
P0 = 101325.0      # atmospheric pressure [Pa]

In [ ]:
def pressure_at_depth(depth_m):
    """Return absolute pressure in pascals at a given depth."""
    return P0 + rho * g * depth_m


def pressure_bar(depth_m):
    """Return absolute pressure in bar."""
    return pressure_at_depth(depth_m) / 100_000

## Gas volume as a function of depth

If the gas occupies volume $V_0$ at the surface, then:

In [ ]:
def gas_volume_at_depth(depth_m, surface_volume=1.0):
    """Return gas volume at depth using Boyle's law.

    The result has the same volume unit as surface_volume.
    """
    return surface_volume * P0 / pressure_at_depth(depth_m)

## First experiment

Start with a gas volume of 1.0 L at the surface.

What happens at 10, 20, 30, and 40 m?

In [ ]:
for depth_m in [0, 10, 20, 30, 40]:
    volume_l = gas_volume_at_depth(depth_m, surface_volume=1.0)
    print(f"{depth_m:2d} m → {volume_l:.3f} L")

You should obtain values close to:

- 0 m → 1.000 L
- 10 m → 0.50 L
- 20 m → 0.33 L
- 30 m → 0.25 L
- 40 m → 0.20 L

This is not a linear relationship.

## Pressure is linear, volume is hyperbolic

Pressure behaves approximately as:

$$
P(z)=P_0+\rho gz
$$

Gas volume behaves as:

$$
V(z)=V_0\frac{P_0}{P_0+\rho gz}
$$

So depth appears in the denominator.

This is why gas volume changes rapidly near the surface and more gradually at greater depth.

In [ ]:
depth = np.linspace(0, 40, 300)
pressure = pressure_bar(depth)
volume_fraction = gas_volume_at_depth(depth, surface_volume=1.0)

In [ ]:
plt.plot(depth, volume_fraction)

plt.xlabel("Depth [m]")
plt.ylabel("Gas volume / surface volume")
plt.title("Relative gas volume vs depth")
plt.grid(True)

plt.show()

## The first 10 meters matter most

Look at the volume changes over equal 10 m intervals.

From the surface to 10 m:

- pressure changes from about 1 bar to 2 bar;
- gas volume changes from 100% to about 50%;
- the volume decreases by about **50% of the surface volume**.

From 10 m to 20 m:

- pressure changes from about 2 bar to 3 bar;
- gas volume changes from about 50% to about 33%;
- the decrease is only about **17% of the surface volume**.

The same 10 m change in depth does not produce the same change in gas volume.

In [ ]:
depths = np.array([0, 10, 20, 30, 40], dtype=float)
volumes = gas_volume_at_depth(depths)

for d, v in zip(depths, volumes):
    print(f"{d:4.0f} m → {100*v:5.1f}% of surface volume")

## Descent: absolute volume change by interval

Let's quantify the amount of volume lost over each 10 m descent interval.

In [ ]:
absolute_changes = np.diff(volumes)

for start, end, delta in zip(depths[:-1], depths[1:], absolute_changes):
    print(
        f"{start:2.0f} → {end:2.0f} m: "
        f"{delta:+.3f} surface-volume units"
    )

The largest absolute loss of gas volume occurs in the first 10 m.

This is a direct consequence of Boyle's law combined with the pressure-depth relationship.

## Ascent is especially important near the surface

The same physics works in reverse.

Suppose a gas volume is 1.0 L at 10 m, where ambient pressure is about 2 bar.

If it rises to the surface at constant temperature:

$$
V_{\text{surface}}
=
1.0 \times \frac{2}{1}
=
2.0\ \text{L}
$$

So the gas volume approximately doubles.

This is why the final part of an ascent is associated with the largest **relative gas expansion**.

In [ ]:
def gas_volume_between_depths(volume_start, depth_start_m, depth_end_m):
    """Return final gas volume after moving between two depths."""
    p_start = pressure_at_depth(depth_start_m)
    p_end = pressure_at_depth(depth_end_m)
    return volume_start * p_start / p_end

In [ ]:
examples = [
    (1.0, 30, 20),
    (1.0, 20, 10),
    (1.0, 10, 0),
]

for v0, d0, d1 in examples:
    v1 = gas_volume_between_depths(v0, d0, d1)
    relative_change = (v1 / v0 - 1) * 100
    print(
        f"{d0:2d} → {d1:2d} m: "
        f"{v0:.2f} → {v1:.2f} L "
        f"({relative_change:+.1f}%)"
    )

## A useful control-systems interpretation

This notebook introduces an important nonlinear effect.

A fixed change in depth does not produce a fixed change in gas volume.

Near the surface:

- pressure is lower;
- the same pressure change is large relative to the current pressure;
- gas volume is therefore more sensitive to depth changes.

This sensitivity will later matter when we model buoyancy and positive feedback during ascent.

## Optional: local sensitivity

For normalized surface volume,

$$
V(z)=\frac{P_0}{P_0+\rho gz}
$$

Its derivative with respect to depth is:

$$
\frac{dV}{dz}
=
-\frac{P_0\rho g}{(P_0+\rho gz)^2}
$$

The magnitude of this derivative decreases with depth.

So mathematically, the volume-depth curve is steepest near the surface.

In [ ]:
def volume_sensitivity(depth_m):
    """Return dV/dz for normalized surface volume V0 = 1."""
    return -(P0 * rho * g) / (P0 + rho * g * depth_m) ** 2

In [ ]:
sensitivity = volume_sensitivity(depth)

plt.plot(depth, sensitivity)

plt.xlabel("Depth [m]")
plt.ylabel("dV/dz [per meter]")
plt.title("Local sensitivity of gas volume to depth")
plt.grid(True)

plt.show()

## Interpretation

Two statements are now both true:

1. **Pressure increases approximately linearly with depth.**
2. **Gas volume changes nonlinearly with depth.**

The first meters are therefore disproportionately important for gas-volume changes.

This is one of the physical foundations of buoyancy instability during ascent.

## Exercises

### 1. A 2 L air space

A flexible air space has a volume of 2.0 L at the surface.

Calculate its volume at:

- 10 m
- 20 m
- 30 m

Use `gas_volume_at_depth()`.

### 2. From 30 m to 10 m

A gas-filled space has a volume of 1.5 L at 30 m.

What is its volume at 10 m?

Use `gas_volume_between_depths()`.

### 3. Find the half-volume depth

At what depth is a gas volume approximately half its surface value?

First estimate mentally.

Then use Python to find the depth numerically.

### 4. Compare equal ascent intervals

Start with 1.0 L at:

- 30 m and ascend to 20 m;
- 20 m and ascend to 10 m;
- 10 m and ascend to 0 m.

Calculate the percentage expansion in each interval.

Which interval produces the largest relative expansion?

## Challenge — Numerical derivative

Estimate the derivative

$$
\frac{dV}{dz}
$$

numerically using NumPy.

Compare your result with the analytical expression:

$$
\frac{dV}{dz}
=
-\frac{P_0\rho g}{(P_0+\rho gz)^2}
$$

Do the two curves agree?

In [ ]:
# Hint:
# numerical_dV_dz = np.gradient(volume_fraction, depth)

# Your code here

## Summary

In this lab we learned that:

- Boyle's law relates pressure and gas volume through $PV=\text{constant}$;
- absolute pressure must be used;
- gas volume decreases nonlinearly during descent;
- gas volume expands nonlinearly during ascent;
- the largest relative volume changes occur near the surface;
- this nonlinear behavior will become important when we study buoyancy feedback.

### Next

In the next notebook, we can connect gas-volume changes to **buoyant force** and ask:

> How can expansion during ascent create a positive feedback loop?